# LLM Fiscal and Monetary Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [2]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')

from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

#### Set up llm

In [2]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [3]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [4]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

## Load data

In [6]:
data_dir = Path('/data/home/xiong/data/Fund/CSR/Traction/output/finetuning')

In [7]:
# prepare files for different tasks
df_train = pd.read_excel(data_dir / 'monetary/cv/training_2.xlsx')
df_test = pd.read_excel(data_dir / 'monetary/cv/testing_2.xlsx')
df_train_agree = df_train[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
df_test_agree = df_test[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

In [8]:
for col in ['agreement_general', 'disagreement_areas']:
    print(df_train_agree[col].value_counts(normalize=True))
    print(df_test_agree[col].value_counts(normalize=True))
    print('\n')

agreement_general
mostly agree           0.735931
disagreement exists    0.190476
irrelevant             0.073593
Name: proportion, dtype: float64
agreement_general
mostly agree           0.568966
disagreement exists    0.327586
irrelevant             0.103448
Name: proportion, dtype: float64


disagreement_areas
Future Policy Direction                                       0.478261
Current Policy Stance                                         0.108696
Monetary Policy Operations                                    0.065217
Monetary Policy Framework                                     0.065217
Central Bank Communication                                    0.065217
Exchange Rate Policy                                          0.043478
Current Policy Stance; Future Policy Direction                0.043478
Economic Assessment                                           0.043478
Economic Assessment; Future Policy Direction                  0.021739
Monetary Policy Operations; Institutions      

In [9]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'monetary_agreement_simple'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_dir = Path('../../src/Traction/prompts')
# Load prompt template
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [11]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'STAFF_TEXT': 'staff',
    'AUTHORITY_TEXT': 'buff'
}

In [71]:
# combine both traing and testing data
sample_df = pd.concat([df_train_agree, df_test_agree])
#sample_df = df_train_agree

In [72]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    sample_df,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [ ]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:1000]}...")


Generated 289 batch messages
Sample IDs: ['332', '319', '419', '12', '25', '431', '323', '45', '55', '320', '73', '131', '284', '118', '420', '79', '77', '407', '19', '324', '287', '389', '294', '317', '23', '4', '110', '78', '393', '97', '423', '87', '349', '50', '303', '330', '354', '386', '366', '370', '103', '91', '350', '337', '96', '351', '361', '371', '378', '316', '33', '434', '410', '63', '39', '408', '346', '26', '128', '127', '355', '292', '328', '380', '429', '29', '281', '427', '129', '375', '93', '47', '101', '68', '326', '123', '356', '104', '70', '0', '130', '304', '278', '336', '403', '362', '301', '402', '415', '88', '9', '432', '385', '416', '412', '1', '384', '279', '106', '341', '288', '114', '83', '18', '390', '437', '306', '52', '369', '24', '363', '36', '59', '66', '396', '305', '290', '340', '395', '359', '57', '382', '37', '376', '3', '27', '71', '14', '430', '352', '409', '343', '89', '49', '117', '421', '291', '360', '313', '15', '99', '109', '367', '392', '

In [74]:
try:
    response = llm_agent.get_response_content(batch_messages[15], reasoning_effort='low', response_format=response_model)
    print(response)
except Exception as e:
    print(f"Error: {e}")

agreement='mostly agree' disagreement_areas=''


In [21]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5',
    temperature=1
)


In [76]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=2000,
        reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 289/289 [04:36<00:00,  1.04msg/s]


In [77]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")
print(merged[0])

Processed 289 responses
{'agreement': 'disagreement exists', 'disagreement_areas': 'Current Policy Stance; Future Policy Direction; Policy Assessment; Economic Assessment', 'id': '332'}


In [79]:
# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

,agreement_llm,disagreement_areas_llm,id
0,disagreement exists,Current Policy Stance; Future Policy Direction...,332
1,mostly agree,,319
2,disagreement exists,Monetary Policy Operations; Future Policy Dire...,419
3,mostly agree,,12
4,mostly agree,,25
...,...,...,...
284,mostly agree,,72
285,mostly agree,,414
286,mostly agree,,42
287,disagreement exists,"Current Policy Stance, Policy Assessment",401


In [ ]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(sample_df, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=['agreement_general', 'agreement_llm'])
print(len(df_merged))
df_merged.head()

289


,agreement_llm,disagreement_areas_llm,id,index,Print ISBN,staff,buff,country,year,agreement_general,disagreement_areas
0,disagreement exists,Current Policy Stance; Future Policy Direction...,332,332,9781484375662,36. Thailand remains resilient in the face of ...,"7. On monetary policy, the authorities have ma...",Thailand,2016,mostly agree,NaN
1,mostly agree,,319,319,9798400244988,"53. Monetary policy has been tightened, with t...",Indonesian authorities acted decisively and wi...,Indonesia,2023,mostly agree,NaN
2,disagreement exists,Monetary Policy Operations; Future Policy Dire...,419,419,9798400243127,47. The SNB’s response to higher inflation has...,Inflation has decreased from a high of 3.5 per...,Switzerland,2023,mostly agree,Exchange Rate Policy
3,mostly agree,,12,12,9781498324410,39. The authorities’ new monetary policy opera...,The NBR has transitioned to the new interest r...,Rwanda,2019,mostly agree,NaN
4,mostly agree,,25,25,9781484338322,38. Keeping inflation on a downward path over ...,Efforts in the monetary area have been key to ...,Uruguay,2017,mostly agree,NaN


In [81]:
# query v6:
print("accuracy Score:")
print(accuracy_score(df_merged['agreement_general'], 
               df_merged['agreement_llm']))

print("f1 Score:")      
print(f1_score(df_merged['agreement_general'], 
         df_merged['agreement_llm'], 
         average='weighted'))

print("Confusion Matrix:")
confusion_matrix(df_merged['agreement_general'], 
                 df_merged['agreement_llm'], 
                 labels=['disagreement exists', 
                         'mostly agree', 
                         'irrelevant'])

accuracy Score:
0.7335640138408305
f1 Score:
0.715466266502525
Confusion Matrix:


array([[ 36,  27,   0],
       [ 30, 173,   0],
       [  2,  18,   3]])

#### Try Stance prediction

In [10]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'monetary_stance_simple'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [17]:
prompt_template

{'system': "You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by {TEXT_AUTHOR}, complete the following two tasks. First, classify the country's recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the {TEXT_AUTHOR}'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy).",
 'schema': 'Resp

In [ ]:
df_train_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df_train[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_train_stance = pd.concat([df_train_stance, df_temp], ignore_index=True)
    
df_test_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df_test[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_test_stance = pd.concat([df_test_stance, df_temp], ignore_index=True)

## merge train and test stance data
df_stance = pd.concat([df_train_stance, df_test_stance], ignore_index=True)


In [14]:
df_stance.head()

,index,Print ISBN,country,year,text,stance_current,stance_future,type
0,332,9781484375662,Thailand,2016,36. Thailand remains resilient in the face of ...,unclear,loosening bias,staff
1,319,9798400244988,Indonesia,2023,"53. Monetary policy has been tightened, with t...",neutral,no change,staff
2,419,9798400243127,Switzerland,2023,47. The SNB’s response to higher inflation has...,restrictive,tightening bias,staff
3,12,9781498324410,Rwanda,2019,39. The authorities’ new monetary policy opera...,accommodative,no change,staff
4,25,9781484338322,Uruguay,2017,38. Keeping inflation on a downward path over ...,unclear,tightening bias,staff


In [15]:
for col in ['stance_current', 'stance_future']:
    print(df_train_stance[col].value_counts(normalize=True))
    print(df_test_stance[col].value_counts(normalize=True))
    print('\n')

stance_current
accommodative    0.367965
unclear          0.357143
restrictive      0.220779
irrelevant       0.038961
neutral          0.015152
Name: proportion, dtype: float64
stance_current
unclear          0.396552
accommodative    0.379310
restrictive      0.163793
irrelevant       0.034483
neutral          0.025862
Name: proportion, dtype: float64


stance_future
no change          0.471861
tightening bias    0.160173
tightening         0.132035
loosening bias     0.101732
unclear            0.054113
loosening          0.043290
irrelevant         0.036797
Name: proportion, dtype: float64
stance_future
no change          0.448276
tightening bias    0.198276
tightening         0.129310
unclear            0.077586
loosening bias     0.060345
loosening          0.051724
irrelevant         0.034483
Name: proportion, dtype: float64




In [16]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'TEXT': 'text',
    'TEXT_AUTHOR': 'type'
}

In [18]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    df_stance,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [19]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:1000]}...")
""

Generated 578 batch messages
Sample IDs: ['332', '319', '419', '12', '25', '431', '323', '45', '55', '320', '73', '131', '284', '118', '420', '79', '77', '407', '19', '324', '287', '389', '294', '317', '23', '4', '110', '78', '393', '97', '423', '87', '349', '50', '303', '330', '354', '386', '366', '370', '103', '91', '350', '337', '96', '351', '361', '371', '378', '316', '33', '434', '410', '63', '39', '408', '346', '26', '128', '127', '355', '292', '328', '380', '429', '29', '281', '427', '129', '375', '93', '47', '101', '68', '326', '123', '356', '104', '70', '0', '130', '304', '278', '336', '403', '362', '301', '402', '415', '88', '9', '432', '385', '416', '412', '1', '384', '279', '106', '341', '288', '114', '83', '18', '390', '437', '306', '52', '369', '24', '363', '36', '59', '66', '396', '305', '290', '340', '395', '359', '57', '382', '37', '376', '3', '27', '71', '14', '430', '352', '409', '343', '89', '49', '117', '421', '291', '360', '313', '15', '99', '109', '367', '392', '

''

In [22]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=2000,
        reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 578/578 [08:41<00:00,  1.11msg/s]


In [23]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")

# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

Processed 578 responses


,stance_current_llm,stance_future_llm,id
0,unclear,loosening,332
1,neutral,no change,319
2,restrictive,tightening bias,419
3,accommodative,no change,12
4,unclear,tightening,25
...,...,...,...
573,restrictive,loosening,72
574,accommodative,tightening,414
575,unclear,unclear,42
576,accommodative,no change,401


In [ ]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(df_stance, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=['stance_current', 'stance_future'])
print(len(df_merged))
df_merged.head()

1156


,stance_current_llm,stance_future_llm,id,index,Print ISBN,country,year,text,stance_current,stance_future,type
0,unclear,loosening,332,332,9781484375662,Thailand,2016,36. Thailand remains resilient in the face of ...,unclear,loosening bias,staff
1,unclear,loosening,332,332,9781484375662,Thailand,2016,"7. On monetary policy, the authorities have ma...",accommodative,no change,buff
2,neutral,no change,319,319,9798400244988,Indonesia,2023,"53. Monetary policy has been tightened, with t...",neutral,no change,staff
3,neutral,no change,319,319,9798400244988,Indonesia,2023,Indonesian authorities acted decisively and wi...,unclear,no change,buff
4,restrictive,tightening bias,419,419,9798400243127,Switzerland,2023,47. The SNB’s response to higher inflation has...,restrictive,tightening bias,staff


In [ ]:
# query v1:
print("Stance Current - Accuracy:", 
      accuracy_score(df_merged['stance_current'], df_merged['stance_current_llm']))
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'], df_merged['stance_future_llm']))
print("Stance Current - F1 Score:", 
      f1_score(df_merged['stance_current'], df_merged['stance_current_llm'], average='weighted'))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'], df_merged['stance_future_llm'], average='weighted'))

print("\nMerging unclear/irrelevant:")
print("Stance Current - Accuracy:", 
      accuracy_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
                     df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)))
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
                     df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)))
print("Stance Current - F1 Score:", 
      f1_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
               df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
               df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))

Stance Current - Accuracy: 0.6288927335640139
Stance Future - Accuracy: 0.5553633217993079
Stance Current - F1 Score: 0.6248249020197549
Stance Future - F1 Score: 0.5509129743552891

Merging unclear/irrelevant:
Stance Current - Accuracy: 0.6600346020761245
Stance Future - Accuracy: 0.5752595155709342
Stance Current - F1 Score: 0.666676964330069
Stance Future - F1 Score: 0.5762582260740265
